# Predicción de impago

Seguimos la estructura de tu plantilla: leer el CSV, preparar los datos, separar entrenamiento y test, escalar y ajustar una regresión logística con `statsmodels`. **Primero hacemos el modelo completo con las variables numéricas y después los gráficos.** No seleccionamos variables por su p-valor.

`Impago`: **0 = no impago, 1 = impago**. Las columnas que ya están codificadas como `0/1` también entran. Cambia únicamente la ruta del CSV en la celda de lectura. Los resultados reales aparecerán cuando ejecutes el notebook con tu CSV.


## Importamos las librerías

Son las mismas herramientas que aparecen en tu plantilla. Si te falta alguna, puedes instalarla descomentando la primera línea.


In [ ]:
# %pip install pandas numpy matplotlib seaborn scikit-learn statsmodels

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.metrics import precision_recall_curve, roc_curve, roc_auc_score


## Leemos el CSV

Sustituye esta ruta por la de tu archivo. Si utiliza punto y coma para separar las columnas, añade `sep=';'` dentro de `read_csv`. Si utiliza coma decimal, añade también `decimal=','`.


In [ ]:
df_bueno = pd.read_csv(r"C:\Users\User\Reto-05-Grupo-Verde-Oscuro\datasets\df_filtrado.csv")
df_bueno.head(5)


## Nos quedamos con las numéricas

`ID` se conserva en el archivo original para identificar los resultados, pero no es una característica que explique el impago. Las columnas de texto quedan fuera. Incluimos `Prima` y el resto de numéricas; si las cuotas francesa y alemana están en el CSV como números, también entran.


In [ ]:
df = df_bueno.copy()
df = df.select_dtypes(include=['number', 'bool'])
df = df.drop(columns='ID', errors='ignore')
df = df.replace([np.inf, -np.inf], np.nan)
df = df.dropna(subset=['Impago'])
df.info()


In [ ]:
df['Impago'].value_counts()


## Separamos X e y

`X` contiene las variables explicativas e `y` indica si hubo impago. No quitamos variables por tener poca correlación ni por sus p-valores.


In [ ]:
X = df.drop(columns='Impago')
y = df['Impago']


## Categorías de referencia

Tus columnas laborales suman 1 y las civiles también. Si ponemos todas junto al intercepto, `Logit` no puede separar sus efectos y puede dar **matriz singular**. Dejamos una categoría de referencia por bloque: el factor sigue en el modelo, y las demás categorías se comparan con esa referencia. **Este ajuste de codificación no se decide por p-valores.**


In [ ]:
for prefijo in ['Tipo_Jornada_Laboral_', 'Estado_Civil_']:
    columnas = X.filter(like=prefijo).columns
    if len(columnas) > 1 and (X[columnas].sum(axis=1) == 1).all():
        print('Categoría de referencia:', columnas[0])
        X = X.drop(columns=columnas[0])

print('Variables incluidas en el modelo:')
print(X.columns.tolist())


## Entrenamiento y test

Como en tu plantilla, usamos el 80 % para entrenar y el 20 % para comprobar las predicciones. `stratify` mantiene la proporción de impagos. Esta separación supone una fila por cliente o préstamo independiente.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    train_size=0.8,
    random_state=42,
    shuffle=True,
    stratify=y
)


## Rellenamos los datos que faltan

Si faltan valores numéricos, usamos la mediana del entrenamiento. Aplicamos esas mismas medianas al test. Las filas sin `Impago` no sirven para entrenar y se apartaron antes.


In [ ]:
medianas = X_train.median()
X_train = X_train.fillna(medianas)
X_test = X_test.fillna(medianas)


## Escalamos los datos

Usamos `StandardScaler` como en la plantilla. Se ajusta con el entrenamiento y después se aplica al test.


In [ ]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)


In [ ]:
X_test_scaled = scaler.transform(X_test)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)


## Creamos el modelo con statsmodels

Este es el mismo bloque de tu plantilla, adaptado a `Impago`. Añadimos la constante y ajustamos la regresión logística. El resumen muestra los coeficientes, sus p-valores y los intervalos de confianza.


In [ ]:
X_train_scaled = sm.add_constant(X_train_scaled, prepend=True, has_constant='add')

modelo_log_statsm = sm.Logit(endog=y_train, exog=X_train_scaled)
modelo_log_statsm = modelo_log_statsm.fit(disp=False, maxiter=100)

print(modelo_log_statsm.summary())


## Vemos los p-valores del modelo completo

Un coeficiente positivo se asocia con mayor riesgo de impago y uno negativo con menor riesgo. `p < 0,05` indica evidencia de asociación en este modelo. Un p-valor alto, por sí solo, no demuestra que la variable sea inútil. **Aquí no eliminamos ninguna por su p-valor.**

Las variables están escaladas: sus coeficientes representan cambios de una desviación estándar. Antes de interpretar `Prima`, comprueba que no se calculó usando el propio `Impago`.


In [ ]:
tabla_coeficientes = pd.DataFrame({
    'Coeficiente': modelo_log_statsm.params,
    'p_valor': modelo_log_statsm.pvalues
})

tabla_coeficientes.sort_values('p_valor')


## Hacemos la predicción

El modelo devuelve la probabilidad de impago. Como en tu plantilla, usamos un umbral de `0,5`: por encima se predice impago y por debajo no impago.


In [ ]:
X_test_scaled = sm.add_constant(X_test_scaled, has_constant='add')

predicciones = modelo_log_statsm.predict(exog=X_test_scaled)
clasificacion = np.where(predicciones < 0.5, 0, 1)

predicciones.head()


## Comprobamos los resultados

La accuracy indica el porcentaje de aciertos. Mira también el **recall de la clase 1**: de todos los impagos reales, cuántos detectamos. Si hay pocos impagos, una accuracy alta puede ocultar que apenas se detectan.


In [ ]:
acierto = accuracy_score(y_test, clasificacion)
print('Acierto:', round(acierto * 100, 2), '%')
print('Error:', round((1 - acierto) * 100, 2), '%')

print(classification_report(y_test, clasificacion, zero_division=0))


In [ ]:
resultados = pd.DataFrame({
    'Impago_real': y_test,
    'Probabilidad_impago': predicciones,
    'Prediccion_impago': clasificacion
})

if 'ID' in df_bueno.columns:
    resultados.insert(0, 'ID', df_bueno.loc[resultados.index, 'ID'])

resultados.head(10)


# Gráficos · después del modelo completo

Ahora visualizamos lo que hemos calculado. Las relaciones entre variables se dibujan con los datos de entrenamiento; los gráficos de predicción usan el test.


## Gráfico de p-valores

La línea roja marca `0,05`. Las variables cuya barra queda a la derecha tienen p-valor mayor que `0,05`. Los valores muy pequeños se leen mejor en la tabla anterior.


In [ ]:
p_valores = modelo_log_statsm.pvalues.drop('const').sort_values()
p_valores.plot(kind='barh', figsize=(11, 7))

plt.axvline(0.05, color='red', linestyle='--')
plt.xlabel('p-valor')
plt.title('P-valores del modelo completo')
plt.tight_layout()
plt.show()


## Correlaciones

Este mapa muestra relaciones lineales entre las numéricas. Valores cerca de 1 o −1 indican una relación fuerte. Entre predictores pueden señalar información repetida que afecta a sus p-valores.


In [ ]:
df_graficos = df.loc[X_train.index]

plt.figure(figsize=(14, 10))
sns.heatmap(df_graficos.corr(), cmap='coolwarm', center=0, vmin=-1, vmax=1)
plt.title('Correlaciones entre las variables numéricas')
plt.tight_layout()
plt.show()


## Relación de cada variable con el impago

En las variables con pocos valores, mostramos la tasa de impago para cada valor. En las continuas, creamos cinco tramos con `qcut`. Estos grupos describen los datos: **no crean la etiqueta ni cambian el modelo**. También aparecen las categorías que se usaron como referencia.


In [ ]:
for variable in df_graficos.drop(columns='Impago').columns:
    if df_graficos[variable].nunique() <= 5:
        tasa = df_graficos.groupby(variable)['Impago'].mean()
    else:
        grupos = pd.qcut(df_graficos[variable], q=5, duplicates='drop')
        tasa = df_graficos.groupby(grupos, observed=True)['Impago'].mean()

    tasa.plot(kind='bar', figsize=(7, 3))
    plt.title(variable + ' e impago')
    plt.ylabel('Tasa de impago')
    plt.xticks(rotation=35)
    plt.tight_layout()
    plt.show()


## Matriz de confusión

Comparamos la clase real con la predicha. La fila de impago muestra cuántos impagos detectamos y cuántos se nos escaparon.


In [ ]:
c_matrix = confusion_matrix(y_test, clasificacion, labels=[0, 1])

sns.heatmap(c_matrix, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicción: 0 = no impago, 1 = impago')
plt.ylabel('Real: 0 = no impago, 1 = impago')
plt.title('Matriz de confusión')
plt.tight_layout()
plt.show()


## Curva precisión-recall

Como en la plantilla, vemos cómo se relacionan la precisión y el recall al cambiar el umbral. Aquí lo visualizamos; todavía no cambiamos el umbral de `0,5`.


In [ ]:
prec, rec, thresholds = precision_recall_curve(y_test, predicciones)

plt.plot(rec, prec)
plt.xlabel('Recall')
plt.ylabel('Precisión')
plt.title('Curva precisión-recall')
plt.show()


## Curva ROC

La curva compara los impagos detectados con los falsos positivos. El AUC resume la capacidad de ordenar el riesgo: `0,5` corresponde al azar y `1` a una separación perfecta.


In [ ]:
fpr, tpr, threshold = roc_curve(y_test, predicciones)

plt.plot(fpr, tpr)
plt.plot([0, 1], [0, 1], '--', color='gray')
plt.xlabel('Ratio de falsos positivos')
plt.ylabel('Ratio de verdaderos positivos')
plt.title('Curva ROC')
plt.show()

print('ROC-AUC:', round(roc_auc_score(y_test, predicciones), 3))


## Guardamos las predicciones y los p-valores

Se crean dos CSV en la carpeta desde la que ejecutas el notebook. Las predicciones corresponden al test: son las filas que no se utilizaron para entrenar.


In [ ]:
resultados.to_csv('predicciones_impago.csv', index=False)
tabla_coeficientes.to_csv('coeficientes_p_valores.csv')
